In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# PHASE 1 — MOSES DATA PREPARATION FOR BERT-MLM
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import re
from sklearn.model_selection import train_test_split

# --------------------------
# Step 1: Load MOSES dataset
# --------------------------

df = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/FBDD/Data/moses_1L.csv"
)
smiles = df["SMILES"].dropna().unique()

print("Total unique SMILES:", len(smiles))

# -------------------------
# Step 2: Write raw SMILES
# -------------------------

with open("raw_moses.smi", "w") as f:
    for s in smiles:
        f.write(s.strip() + "\n")

print("Saved: raw_moses.smi")

# -------------------------------------------------
# Step 3: Train / Test / Eval split (80 / 10 / 10)
# -------------------------------------------------

train, temp = train_test_split(
    smiles, test_size=0.2, random_state=42
)

test, eval_ = train_test_split(
    temp, test_size=0.5, random_state=42
)

def write_smi(filename, data):
    with open(filename, "w") as f:
        for s in data:
            f.write(s.strip() + "\n")

write_smi("train.smi", train)
write_smi("test.smi", test)
write_smi("eval.smi", eval_)

print("Split sizes:")
print("Train:", len(train))
print("Test :", len(test))
print("Eval :", len(eval_))

# --------------------------------------------------
# Step 4: MOSES-style SMILES tokenizer (TEXT-LEVEL)
# --------------------------------------------------

SMILES_REGEX = re.compile(
    r"\[[^\]]+\]|Br|Cl|Si|Na|Li|Mg|Ca|"
    r"[A-Z][a-z]?|\d+|=|#|-|\(|\)|\.|@|\+|\\|/"
)

def moses_encode(smiles: str):
    """
    Convert SMILES to MOSES-tokenized string
    """
    smiles = smiles.strip()
    if len(smiles) < 2:
        return None

    tokens = SMILES_REGEX.findall(smiles)
    if len(tokens) < 2:
        return None

    return " ".join(tokens)

# ---------------------------------------------
# Step 5: Encode files (encode_moses.py logic)
# ---------------------------------------------

def encode_file(input_file, output_file):
    total = 0
    skipped = 0

    with open(input_file) as fin, open(output_file, "w") as fout:
        for line in fin:
            total += 1
            encoded = moses_encode(line)
            if encoded is None:
                skipped += 1
                continue
            fout.write(encoded + "\n")

    print(f"{input_file} → {output_file}")
    print(f"Total   : {total}")
    print(f"Encoded : {total - skipped}")
    print(f"Skipped : {skipped}")

encode_file("train.smi", "train_encoded_moses.smi")
encode_file("test.smi", "test_encoded_moses.smi")
encode_file("eval.smi", "eval_encoded_moses.smi")

# --------------------------------------------
# Step 6: Sanity check (print first 10 lines)
# --------------------------------------------

def print_head(file, n=10):
    print(f"\n {file} (first {n} lines)")
    with open(file) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.strip())

print_head("train_encoded_moses.smi")
print_head("test_encoded_moses.smi")
print_head("eval_encoded_moses.smi")

print("\n PHASE-1 COMPLETE: Data ready for BERT tokenizer training")


In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# PHASE 2 — Build FragmentBERT Tokenizer
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

!pip install tokenizers
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from tokenizers.normalizers import Sequence # Import Sequence normalizer

import os

# ------
# PATHS
# ------

BASE_DIR = "FragmentBERT"
os.makedirs(BASE_DIR, exist_ok=True)

TRAIN_FILE = "train_encoded_moses.smi"
TEST_FILE  = "test_encoded_moses.smi"
EVAL_FILE  = "eval_encoded_moses.smi"

TOKENIZER_PATH = os.path.join(BASE_DIR, "tokenizer.json")

# ---------------------
# INITIALIZE TOKENIZER
# ---------------------

tokenizer = Tokenizer(
    WordLevel(unk_token="[UNK]")
)

tokenizer.pre_tokenizer = Whitespace()

# Add an explicit empty normalizer to satisfy BertTokenizerFast's requirements
tokenizer.normalizer = Sequence([])

# --------
# TRAINER
# --------

trainer = WordLevelTrainer(
    vocab_size=30000, # A reasonable vocabulary size, e.g., 30000
    min_frequency=2,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[CLS]",
        "[SEP]",
        "[MASK]"
    ]
)

# ----------------
# TRAIN TOKENIZER
# ----------------

tokenizer.train(
    files=[TRAIN_FILE, TEST_FILE, EVAL_FILE],
    trainer=trainer
)

# -----------------------------
# POST PROCESSOR (BERT FORMAT)
# -----------------------------

tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", tokenizer.token_to_id("[CLS]")),
        ("[SEP]", tokenizer.token_to_id("[SEP]"))
    ],
)

# ---------------
# SAVE TOKENIZER
# ---------------

tokenizer.save(TOKENIZER_PATH)

print(" Tokenizer saved at:", TOKENIZER_PATH)
print("Vocab size:", tokenizer.get_vocab_size())
print()

# -----------------
# Verify TOKENIZER
# -----------------

tok = Tokenizer.from_file("FragmentBERT/tokenizer.json")

print(tok.encode("C Nc O c1ccccc1").tokens)
print(tok.encode("C [MASK] c1ccccc1").tokens)
print()

# --------------------------
# Inspect special token IDs
# --------------------------

tokenizer.get_vocab()

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# PHASE 3 — Masked Language Model Pretraining
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# --------
# Imports
# --------
import os
from transformers import BertTokenizerFast, BertConfig, BertForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import load_dataset

# ------
# Paths
# ------
TOKENIZER_PATH = "/content/FragmentBERT/tokenizer.json"
TRAIN_FILE = "train_encoded_moses.smi"
EVAL_FILE = "eval_encoded_moses.smi"
OUTPUT_DIR = "/content/FragmentBERT/checkpoints"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------
# Load Tokenizer
# ---------------
tokenizer = BertTokenizerFast(
    tokenizer_file=TOKENIZER_PATH,
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    unk_token="[UNK]",
    pad_token="[PAD]"
)

# -------------
# Load Dataset
# -------------
# Each line in the file is a molecule's fragment sentence
dataset = load_dataset('text', data_files={'train': TRAIN_FILE, 'validation': EVAL_FILE})

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# -----------------------
# Data Collator (for MLM)
# -----------------------
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# -------------------
# BERT Configuration
# -------------------
config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=256,        # smaller model for quick pretraining
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=512,
)

model = BertForMaskedLM(config)

# -------------------
# Training Arguments
# -------------------
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    logging_steps=100,
    save_total_limit=5,
    learning_rate=5e-4,
    weight_decay=0.01,
    report_to="none"
)

# --------
# Trainer
# --------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
)

# ------
# Train
# ------
trainer.train()

# -----------------
# Save final model
# -----------------
trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "final_tokenizer"))

print(" FragmentBERT MLM pretraining complete!")


In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# PHASE 4.1 — FragmentBERT MLM Optimization
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

!pip install rdkit

# --------
# Imports
# --------
import re
import pandas as pd
from collections import OrderedDict
from transformers import pipeline
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors, Crippen, Lipinski
from rdkit.Chem.QED import qed
from IPython.display import display

# ------------------
# Inp+ut True Smiles
# ------------------
true_smiles = "CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl"
print()
print("True Smiles = ", true_smiles)
print()

# --------------------
# SMILES Tokenization
# --------------------
SMILES_REGEX = re.compile(
    r"\[[^\]]+\]|Br|Cl|Si|Na|Li|Mg|Ca|"
    r"[A-Z][a-z]?|\d+|=|#|-|\(|\)|\.|@|\+|\\|/"
)

def moses_encode(smiles: str):
    smiles = smiles.strip()
    tokens = SMILES_REGEX.findall(smiles)
    return " ".join(tokens) if len(tokens) > 1 else None

# --------------------------
# Drug-likeness Calculation
# --------------------------
def compute_druglikeness(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return {
        "SMILES": smiles,
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotB": Lipinski.NumRotatableBonds(mol),
        "TPSA": Descriptors.TPSA(mol),
        "QED": qed(mol)
    }


# ----------------------
# Load FragmentBERT MLM
# ----------------------
fill_mask = pipeline(
    "fill-mask",
    model="FragmentBERT/checkpoints/final_model",
    tokenizer="FragmentBERT/checkpoints/final_tokenizer"
)

# ------------------------------------
# Core Fragment Optimization Function
# ------------------------------------
def fragment_optimization(
    masked_raw_smiles,
    top_k=40,
    output_csv="results.csv",
    title="Fragment Optimization"
):
    print(f"\n {title}")
    print("-" * 60)

    encoded = moses_encode(masked_raw_smiles)
    masked_input = encoded.replace("X", "[MASK]")

    print("Model input:")
    print(masked_input)

    outputs = fill_mask(masked_input, top_k=top_k)

    unique_mols = OrderedDict()
    for o in outputs:
        smi = o["sequence"].replace(" ", "")
        mol = Chem.MolFromSmiles(smi)
        if mol:
            canon = Chem.MolToSmiles(mol, canonical=True)
            if canon not in unique_mols:
                unique_mols[canon] = o["score"]

    gen_smis = list(unique_mols.keys())
    print(f"\nGenerated valid molecules: {len(gen_smis)}")

    # Drug-likeness filtering
    druglike = []
    for smi in gen_smis:
        props = compute_druglikeness(smi)
        if props and (
            props["MW"] <= 500 and
            props["LogP"] <= 5 and
            props["HBD"] <= 5 and
            props["HBA"] <= 10 and
            props["RotB"] <= 10 and
            props["TPSA"] <= 140 and
            props["QED"] >= 0.4
        ):
            druglike.append(props)

    df = pd.DataFrame(druglike)
    df.to_csv(output_csv, index=False)

    if df.empty:
        print("\n No drug-like molecules passed filters.")
    else:
        print(f"\n Drug-like molecules retained: {len(df)}")
        print(
            df[["SMILES", "MW", "LogP", "HBD", "HBA", "RotB", "TPSA", "QED"]]
            .to_string(index=False)
        )

        # Visualization
        gen_mols = []
        gen_legends = []

        for s in gen_smis:
            m = Chem.MolFromSmiles(s)
            if m is not None:
                gen_mols.append(m)
                gen_legends.append(s)

        if gen_mols:
            img = Draw.MolsToGridImage(
                gen_mols[:12],
                molsPerRow=4,
                legends=gen_legends[:12]
            )
            display(img)
        else:
            print("No valid molecules to visualize.")

    return df

# -------------------------------
# Alkoxy side-chain optimization
# -------------------------------
fragment_optimization(
    masked_raw_smiles=
    "XOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl",
    top_k=30,
    output_csv="druglike_generated_molecules_alkoxy.csv",
    title="Alkoxy Side-Chain Optimization"
)


# ----------------------------------------
# Linker optimization (aryl–alkyl bridge)
# ----------------------------------------
fragment_optimization(
    masked_raw_smiles=
    "CCOXC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl",
    top_k=40,
    output_csv="druglike_generated_molecules_linker.csv",
    title="Linker Optimization"
)

# ---------------------------------
# Aromatic SAR (ring substitution)
# ---------------------------------
fragment_optimization(
    masked_raw_smiles=
    "CCOC1=CC=X(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl",
    top_k=40,
    output_csv="druglike_generated_molecules_aromatic_sar.csv",
    title="Aromatic SAR Exploration"
)


# ---------------------------------
# Halogen scan (Cl → bioisosteres)
# ---------------------------------
fragment_optimization(
    masked_raw_smiles=
    "CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)X",
    top_k=20,
    output_csv="druglike_generated_molecules_halogen.csv",
    title="Halogen Replacement"
)


print("\n PHASE 4.1 COMPLETE — FragmentBERT MLM Optimization successful")

In [ ]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# PHASE 4.2 — Comprehensive Chemical Substitution Analysis
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import json
import os
import math
from collections import OrderedDict, defaultdict, deque
import random
from datetime import datetime
from rdkit import Chem
from rdkit.Chem import Draw, AllChem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem.QED import qed
from rdkit.Chem import rdMolDescriptors
from IPython.display import display, HTML, Image
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------
# Input True Smiles
# ------------------
true_smiles = "CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl"
print()
print("Original SMILES = ", true_smiles)
print()

# --------------------------
# Drug-likeness Calculation
# --------------------------
def compute_all_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Basic properties
    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)
    rotb = Lipinski.NumRotatableBonds(mol)
    tpsa = Descriptors.TPSA(mol)
    qed_val = qed(mol)

    # Additional properties
    ring_count = rdMolDescriptors.CalcNumRings(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    heavy_atoms = mol.GetNumHeavyAtoms()

    # Calculate violations of Lipinski's Rule of 5
    violations = 0
    if mw > 500: violations += 1
    if logp > 5: violations += 1
    if hbd > 5: violations += 1
    if hba > 10: violations += 1

    # Calculate synthetic accessibility score (simplified)
    synth_score = 1.0 - min(1.0, (mw/1000 + ring_count/10 + rotb/20))

    return {
        "SMILES": smiles,
        "MW": mw,
        "LogP": logp,
        "HBD": hbd,
        "HBA": hba,
        "RotB": rotb,
        "TPSA": tpsa,
        "QED": qed_val,
        "Heavy_Atoms": heavy_atoms,
        "Ring_Count": ring_count,
        "Aromatic_Rings": aromatic_rings,
        "Lipinski_Violations": violations,
        "Synth_Score": synth_score,
        "Druglike": violations <= 1 and 200 <= mw <= 500 and -2 <= logp <= 5
    }

# -------------------------------------
# COMPREHENSIVE SUBSTITUTION GENERATOR
# -------------------------------------
def generate_all_possible_substitutions(original_smiles):
    """Generate ALL possible functional group substitutions"""

    print(f"\n{'='*100}")
    print("GENERATING ALL POSSIBLE SUBSTITUTIONS")
    print(f"{'='*100}")

    mol = Chem.MolFromSmiles(original_smiles)
    if not mol:
        print("Failed to parse SMILES")
        return []

    print(f"Analyzing molecule with {mol.GetNumAtoms()} atoms...")

    # Define ALL possible substitution pairs for different atom types
    SUBSTITUTION_LIBRARY = {
        # Chlorine substitutions (most common)
        17: ['F', 'Br', 'I', 'OH', 'NH2', 'SH', 'CN', 'CF3', 'CH3', 'OCH3', 'SCH3'],

        # Oxygen substitutions (hydroxyl groups)
        8: ['F', 'Cl', 'Br', 'NH2', 'SH', 'CH3', 'CF3', 'CN', 'OCH3', 'SCH3', 'H'],

        # Carbon substitutions (terminal carbons)
        6: ['F', 'Cl', 'Br', 'OH', 'NH2', 'SH', 'CN', 'CF3', 'CH3', 'O', 'N'],

        # Nitrogen substitutions
        7: ['O', 'C', 'CH', 'CH2', 'CH3', 'OH', 'F', 'Cl'],

        # Fluorine substitutions
        9: ['Cl', 'Br', 'I', 'OH', 'NH2', 'CH3', 'CF3', 'H'],

        # Bromine substitutions
        35: ['Cl', 'F', 'I', 'OH', 'NH2', 'CH3', 'CF3'],

        # Sulfur substitutions
        16: ['O', 'NH', 'CH2', 'CF2', 'Se']
    }

    all_results = []
    attempted_count = 0
    successful_count = 0

    # Try substitutions on ALL atoms
    for atom_idx in range(mol.GetNumAtoms()):
        atom = mol.GetAtomWithIdx(atom_idx)
        atomic_num = atom.GetAtomicNum()
        symbol = atom.GetSymbol()

        if atomic_num in SUBSTITUTION_LIBRARY:
            possible_subs = SUBSTITUTION_LIBRARY[atomic_num]

            for new_symbol in possible_subs:
                attempted_count += 1

                # Skip self-substitution
                if new_symbol == symbol:
                    continue

                # Try the substitution
                new_smiles = perform_rdkit_substitution_fixed(mol, atom_idx, new_symbol)

                if new_smiles and new_smiles != original_smiles:
                    # Calculate properties
                    props = compute_all_properties(new_smiles)
                    if props:
                        successful_count += 1

                        # Calculate improvements
                        orig_props = compute_all_properties(original_smiles)
                        qed_improvement = props['QED'] - orig_props['QED']

                        # Create result entry
                        result = {
                            'SMILES': new_smiles,
                            'Substitution': f"Atom {atom_idx} ({symbol}) → {new_symbol}",
                            'Atom_Index': atom_idx,
                            'Original_Atom': symbol,
                            'New_Group': new_symbol,
                            'QED': props['QED'],
                            'QED_Improvement': qed_improvement,
                            'LogP': props['LogP'],
                            'MW': props['MW'],
                            'HBD': props['HBD'],
                            'HBA': props['HBA'],
                            'RotB': props['RotB'],
                            'TPSA': props['TPSA'],
                            'Druglike': props['Druglike'],
                            'Synth_Score': props['Synth_Score'],
                            'Heavy_Atoms': props['Heavy_Atoms']
                        }

                        all_results.append(result)

                # Progress indicator
                if attempted_count % 50 == 0:
                    print(f"  Attempted {attempted_count} substitutions...")

    print(f"\nSubstitution generation complete!")
    print(f"Attempted: {attempted_count} substitutions")
    print(f"Successful: {successful_count} valid molecules")
    print(f"Unique molecules: {len(all_results)}")

    return all_results

def perform_rdkit_substitution_fixed(mol, atom_idx, new_symbol):
    """Perform substitution with comprehensive error handling"""
    try:
        emol = Chem.RWMol(mol)
        atom = emol.GetAtomWithIdx(atom_idx)

        # Store original atom properties
        original_num = atom.GetAtomicNum()

        # Handle different substitution types
        if new_symbol == 'F':
            atom.SetAtomicNum(9)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'Cl':
            atom.SetAtomicNum(17)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'Br':
            atom.SetAtomicNum(35)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'I':
            atom.SetAtomicNum(53)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'OH':
            atom.SetAtomicNum(8)
            atom.SetNumExplicitHs(1)
        elif new_symbol == 'NH2':
            atom.SetAtomicNum(7)
            atom.SetNumExplicitHs(2)
        elif new_symbol == 'SH':
            atom.SetAtomicNum(16)
            atom.SetNumExplicitHs(1)
        elif new_symbol == 'CN':
            # Add cyano group
            new_c = Chem.Atom(6)
            new_n = Chem.Atom(7)
            c_idx = emol.AddAtom(new_c)
            n_idx = emol.AddAtom(new_n)
            emol.AddBond(atom_idx, c_idx, Chem.BondType.SINGLE)
            emol.AddBond(c_idx, n_idx, Chem.BondType.TRIPLE)
        elif new_symbol == 'CF3':
            # Replace with carbon and add 3 fluorines
            atom.SetAtomicNum(6)
            for _ in range(3):
                f_atom = Chem.Atom(9)
                f_idx = emol.AddAtom(f_atom)
                emol.AddBond(atom_idx, f_idx, Chem.BondType.SINGLE)
        elif new_symbol == 'CH3':
            atom.SetAtomicNum(6)
            atom.SetNumExplicitHs(3)
        elif new_symbol == 'OCH3':
            atom.SetAtomicNum(8)
            # Add methyl group
            new_c = Chem.Atom(6)
            c_idx = emol.AddAtom(new_c)
            emol.AddBond(atom_idx, c_idx, Chem.BondType.SINGLE)
        elif new_symbol == 'SCH3':
            atom.SetAtomicNum(16)
            # Add methyl group
            new_c = Chem.Atom(6)
            c_idx = emol.AddAtom(new_c)
            emol.AddBond(atom_idx, c_idx, Chem.BondType.SINGLE)
        elif new_symbol == 'O':
            atom.SetAtomicNum(8)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'N':
            atom.SetAtomicNum(7)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'C':
            atom.SetAtomicNum(6)
            atom.SetNumExplicitHs(0)
        elif new_symbol == 'H':
            # Remove atom (replace with hydrogen)
            # This is complex - skip for now
            return None
        elif new_symbol == 'Se':
            atom.SetAtomicNum(34)  # Selenium
            atom.SetNumExplicitHs(0)
        else:
            # Unknown symbol
            return None

        atom.UpdatePropertyCache()
        Chem.SanitizeMol(emol)
        new_smiles = Chem.MolToSmiles(emol)

        # Validate the new molecule
        if Chem.MolFromSmiles(new_smiles):
            return new_smiles
        else:
            return None

    except Exception as e:
        # Silent error - just return None
        return None

# --------------------------------------
# COMPREHENSIVE VISUALIZATION FUNCTIONS
# --------------------------------------
def visualize_all_structures_grid(df, orig_smiles, max_display=36):
    """Create a grid visualization of all structures"""

    print(f"\n{'='*100}")
    print(f"VISUALIZING ALL {min(len(df), max_display)} STRUCTURES")
    print(f"{'='*100}")

    if df.empty:
        print("No structures to visualize")
        return

    # Sort by QED score
    df_sorted = df.sort_values('QED', ascending=False)

    # Prepare molecules for display
    mols_to_display = []
    legends = []

    # Add original molecule
    orig_mol = Chem.MolFromSmiles(orig_smiles)
    if orig_mol:
        orig_props = compute_all_properties(orig_smiles)
        mols_to_display.append(orig_mol)
        legends.append(f"ORIGINAL\nQED: {orig_props['QED']:.3f}")

    # Add all other molecules (up to max_display)
    display_count = min(len(df_sorted), max_display - 1)  # -1 for original

    for i, (_, row) in enumerate(df_sorted.head(display_count).iterrows()):
        mol = Chem.MolFromSmiles(row['SMILES'])
        if mol:
            mols_to_display.append(mol)

            # Create informative legend
            legend = f"{row['Substitution']}\n"
            legend += f"QED: {row['QED']:.3f} (Δ{row['QED_Improvement']:+.3f})\n"
            legend += f"MW: {row['MW']:.0f} | LogP: {row['LogP']:.1f}"

            if not row['Druglike']:
                legend += "\n⚠️ Not druglike"

            legends.append(legend)

    # Calculate grid dimensions
    n_mols = len(mols_to_display)
    if n_mols == 0:
        print("No valid molecules to display")
        return

    # Determine grid layout
    if n_mols <= 4:
        cols = 2
    elif n_mols <= 9:
        cols = 3
    elif n_mols <= 16:
        cols = 4
    else:
        cols = 6

    rows = math.ceil(n_mols / cols)

    print(f"Displaying {n_mols} molecules in {rows}x{cols} grid")

    # Create grid image
    try:
        img = Draw.MolsToGridImage(
            mols_to_display,
            molsPerRow=cols,
            subImgSize=(300, 300),
            legends=legends,
            returnPNG=False
        )
        display(img)

    except Exception as e:
        print(f"Error creating grid image: {e}")
        # Fallback: display in smaller batches
        display_in_batches(mols_to_display, legends, batch_size=12)

def display_in_batches(mols, legends, batch_size=12):
    """Display molecules in smaller batches"""
    for i in range(0, len(mols), batch_size):
        batch_mols = mols[i:i+batch_size]
        batch_legends = legends[i:i+batch_size]

        cols = 4 if len(batch_mols) >= 4 else 2

        print(f"\nBatch {i//batch_size + 1}:")
        img = Draw.MolsToGridImage(
            batch_mols,
            molsPerRow=cols,
            subImgSize=(250, 250),
            legends=batch_legends,
            returnPNG=False
        )
        display(img)

def create_qed_distribution_plot(df, orig_qed):
    """Create detailed QED distribution plot"""

    if df.empty:
        return

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('QED Distribution', 'QED vs Molecular Weight',
                       'QED vs LogP', 'Improvement Analysis'),
        specs=[[{'type': 'histogram'}, {'type': 'scatter'}],
               [{'type': 'scatter'}, {'type': 'bar'}]]
    )

    # 1. QED Distribution Histogram
    fig.add_trace(
        go.Histogram(
            x=df['QED'],
            nbinsx=30,
            name='QED Distribution',
            marker_color='skyblue',
            opacity=0.7
        ),
        row=1, col=1
    )

    # Add original QED line
    fig.add_vline(x=orig_qed, line_dash="dash", line_color="red",
                  annotation_text="Original", row=1, col=1)

    # 2. QED vs Molecular Weight
    fig.add_trace(
        go.Scatter(
            x=df['MW'],
            y=df['QED'],
            mode='markers',
            name='Molecules',
            marker=dict(
                size=8,
                color=df['QED_Improvement'],
                colorscale='RdYlGn',
                showscale=True,
                colorbar=dict(title="ΔQED"),
                line=dict(width=1, color='DarkSlateGrey')
            ),
            text=df['Substitution'],
            hovertemplate='<b>%{text}</b><br>' +
                         'MW: %{x:.1f}<br>' +
                         'QED: %{y:.3f}<br>' +
                         'ΔQED: %{marker.color:.3f}<extra></extra>'
        ),
        row=1, col=2
    )

    # Add druglike region
    fig.add_vrect(x0=200, x1=500, fillcolor="green", opacity=0.1,
                  layer="below", line_width=0, row=1, col=2)

    # 3. QED vs LogP
    fig.add_trace(
        go.Scatter(
            x=df['LogP'],
            y=df['QED'],
            mode='markers',
            name='Molecules',
            marker=dict(
                size=8,
                color=df['MW'],
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title="MW"),
                line=dict(width=1, color='DarkSlateGrey')
            ),
            text=df['Substitution'],
            hovertemplate='<b>%{text}</b><br>' +
                         'LogP: %{x:.2f}<br>' +
                         'QED: %{y:.3f}<br>' +
                         'MW: %{marker.color:.1f}<extra></extra>'
        ),
        row=2, col=1
    )

    # Add optimal LogP region
    fig.add_vrect(x0=1, x1=3, fillcolor="yellow", opacity=0.1,
                  layer="below", line_width=0, row=2, col=1)

    # 4. Top 10 improvements
    top_10 = df.nlargest(10, 'QED_Improvement')
    fig.add_trace(
        go.Bar(
            x=top_10['Substitution'],
            y=top_10['QED_Improvement'],
            name='Top Improvements',
            marker_color=px.colors.qualitative.Set3[:10],
            text=top_10['QED_Improvement'].round(3),
            textposition='auto'
        ),
        row=2, col=2
    )

    # Update layout
    fig.update_layout(
        height=800,
        showlegend=False,
        title_text="Comprehensive Analysis of All Generated Molecules",
        title_font_size=20
    )

    # Update axes labels
    fig.update_xaxes(title_text="QED Score", row=1, col=1)
    fig.update_yaxes(title_text="Count", row=1, col=1)

    fig.update_xaxes(title_text="Molecular Weight", row=1, col=2)
    fig.update_yaxes(title_text="QED Score", row=1, col=2)

    fig.update_xaxes(title_text="LogP", row=2, col=1)
    fig.update_yaxes(title_text="QED Score", row=2, col=1)

    fig.update_xaxes(title_text="Substitution", row=2, col=2, tickangle=45)
    fig.update_yaxes(title_text="QED Improvement (ΔQED)", row=2, col=2)

    fig.show()

def create_interactive_scatter_plot(df, orig_props):
    """Create interactive scatter plot of all molecules"""

    if df.empty:
        return

    # Create classification
    conditions = [
        (df['QED'] >= 0.7),
        (df['QED'] >= 0.6) & (df['QED'] < 0.7),
        (df['QED'] >= 0.5) & (df['QED'] < 0.6),
        (df['QED'] >= 0.4) & (df['QED'] < 0.5),
        (df['QED'] < 0.4)
    ]
    choices = ['Excellent (≥0.7)', 'Good (0.6-0.7)', 'Moderate (0.5-0.6)',
               'Poor (0.4-0.5)', 'Very Poor (<0.4)']
    df['QED_Class'] = np.select(conditions, choices, default='Unclassified')

    # Create figure
    fig = px.scatter(
        df,
        x='MW',
        y='QED',
        color='QED_Class',
        size='Synth_Score',
        hover_name='Substitution',
        hover_data={
            'LogP': ':.2f',
            'QED_Improvement': ':.3f',
            'HBD': True,
            'HBA': True,
            'Druglike': True,
            'QED_Class': False,
            'Synth_Score': False
        },
        title=f"All Generated Molecules (n={len(df)})",
        color_discrete_map={
            'Excellent (≥0.7)': '#2ECC71',  # Green
            'Good (0.6-0.7)': '#3498DB',    # Blue
            'Moderate (0.5-0.6)': '#F39C12', # Orange
            'Poor (0.4-0.5)': '#E74C3C',    # Red
            'Very Poor (<0.4)': '#7D3C98'   # Purple
        }
    )

    # Add original molecule
    fig.add_trace(
        go.Scatter(
            x=[orig_props['MW']],
            y=[orig_props['QED']],
            mode='markers',
            name='Original',
            marker=dict(
                size=20,
                symbol='star',
                color='black',
                line=dict(width=2, color='white')
            ),
            text=['Original Molecule'],
            hovertemplate='<b>Original Molecule</b><br>' +
                         'MW: %{x:.1f}<br>' +
                         'QED: %{y:.3f}<extra></extra>'
        )
    )

    # Add druglike region
    fig.add_shape(
        type="rect",
        x0=200, y0=0,
        x1=500, y1=1,
        fillcolor="LightGreen",
        opacity=0.1,
        layer="below",
        line_width=0,
    )

    fig.add_annotation(
        x=350, y=0.95,
        text="Druglike Region (MW 200-500)",
        showarrow=False,
        font=dict(size=10, color="darkgreen")
    )

    # Update layout
    fig.update_layout(
        height=600,
        xaxis_title="Molecular Weight",
        yaxis_title="QED Score",
        legend_title="QED Classification",
        hovermode='closest'
    )

    fig.show()

def create_property_radar_charts(df, orig_props, top_n=10):
    """Create radar charts for top molecules"""

    if df.empty:
        return

    # Get top molecules
    top_molecules = df.nlargest(top_n, 'QED')

    # Properties to display in radar chart
    properties = ['QED', 'Druglike_Score', 'Synth_Score', 'LogP_Score', 'Size_Score']

    # Prepare data
    radar_data = []

    # Add original
    orig_scores = {
        'QED': orig_props['QED'],
        'Druglike_Score': 1.0 if orig_props['Druglike'] else 0.0,
        'Synth_Score': orig_props['Synth_Score'],
        'LogP_Score': max(0, 1 - abs(orig_props['LogP'] - 2) / 5),  # Optimal around 2
        'Size_Score': max(0, 1 - abs(orig_props['MW'] - 350) / 200)  # Optimal around 350
    }
    radar_data.append(('Original', orig_scores))

    # Add top molecules
    for i, (_, row) in enumerate(top_molecules.iterrows()):
        scores = {
            'QED': row['QED'],
            'Druglike_Score': 1.0 if row['Druglike'] else 0.0,
            'Synth_Score': row['Synth_Score'],
            'LogP_Score': max(0, 1 - abs(row['LogP'] - 2) / 5),
            'Size_Score': max(0, 1 - abs(row['MW'] - 350) / 200)
        }
        radar_data.append((f"#{i+1}: {row['Substitution'][:20]}", scores))

    # Create radar charts
    n_charts = min(4, len(radar_data))  # Show max 4 charts
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[name for name, _ in radar_data[:n_charts]],
        specs=[[{'type': 'polar'}, {'type': 'polar'}],
               [{'type': 'polar'}, {'type': 'polar'}]]
    )

    colors = ['red', 'blue', 'green', 'orange']

    for i, (name, scores) in enumerate(radar_data[:n_charts]):
        row = (i // 2) + 1
        col = (i % 2) + 1

        fig.add_trace(
            go.Scatterpolar(
                r=[scores[prop] for prop in properties],
                theta=properties,
                fill='toself',
                name=name,
                line_color=colors[i],
                opacity=0.7
            ),
            row=row, col=col
        )

    fig.update_layout(
        height=600,
        showlegend=True,
        title_text=f"Property Radar Charts - Top {min(top_n, len(top_molecules))} Molecules",
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )
        ),
        polar2=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )
        ),
        polar3=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )
        ),
        polar4=dict(
            radialaxis=dict(
                visible=True,
                range=[0, 1]
            )
        )
    )

    fig.show()

def create_improvement_heatmap(df):
    """Create heatmap of substitution improvements"""

    if df.empty:
        return

    # Extract substitution patterns
    df['Pattern'] = df.apply(lambda row: f"{row['Original_Atom']} → {row['New_Group']}", axis=1)

    # Create pivot table
    pivot_data = df.pivot_table(
        values='QED_Improvement',
        index='Original_Atom',
        columns='New_Group',
        aggfunc='mean'
    ).fillna(0)

    # Create heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(
        pivot_data,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=0,
        square=True,
        cbar_kws={'label': 'Average ΔQED'}
    )

    plt.title('Average QED Improvement by Substitution Pattern', fontsize=16)
    plt.xlabel('New Group', fontsize=12)
    plt.ylabel('Original Atom', fontsize=12)
    plt.tight_layout()
    plt.show()

def create_detailed_statistics_table(df, orig_props):
    """Create detailed statistics table"""

    if df.empty:
        return

    print(f"\n{'='*100}")
    print("DETAILED STATISTICS FOR ALL GENERATED MOLECULES")
    print(f"{'='*100}")

    # Basic statistics
    print(f"\nTotal molecules generated: {len(df)}")
    print(f"Original QED: {orig_props['QED']:.3f}")
    print(f"Best QED: {df['QED'].max():.3f} (Improvement: {df['QED'].max() - orig_props['QED']:+.3f})")
    print(f"Worst QED: {df['QED'].min():.3f} (Decline: {df['QED'].min() - orig_props['QED']:+.3f})")
    print(f"Average QED: {df['QED'].mean():.3f}")

    # Improvement statistics
    improvements = df[df['QED_Improvement'] > 0]
    declines = df[df['QED_Improvement'] < 0]

    print(f"\nImprovements (ΔQED > 0): {len(improvements)} molecules ({len(improvements)/len(df)*100:.1f}%)")
    print(f"Declines (ΔQED < 0): {len(declines)} molecules ({len(declines)/len(df)*100:.1f}%)")
    print(f"Neutral (ΔQED = 0): {len(df) - len(improvements) - len(declines)} molecules")

    # Druglike statistics
    druglike = df[df['Druglike']]
    print(f"\nDruglike molecules: {len(druglike)} ({len(druglike)/len(df)*100:.1f}%)")

    # QED classification
    print(f"\nQED Score Distribution:")
    qed_bins = [(0.0, 0.4), (0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 1.0)]
    labels = ['Very Poor (<0.4)', 'Poor (0.4-0.5)', 'Moderate (0.5-0.6)',
              'Good (0.6-0.7)', 'Excellent (≥0.7)']

    for (low, high), label in zip(qed_bins, labels):
        count = len(df[(df['QED'] >= low) & (df['QED'] < high)])
        if high == 1.0:  # Include upper bound for excellent
            count = len(df[df['QED'] >= low])
        percentage = count / len(df) * 100
        print(f"  {label}: {count} molecules ({percentage:.1f}%)")

    # Top 10 molecules table
    print(f"\n{'='*100}")
    print("TOP 10 MOLECULES BY QED SCORE")
    print(f"{'='*100}")
    print(f"{'Rank':<6} {'Substitution':<25} {'QED':<8} {'ΔQED':<8} {'MW':<8} {'LogP':<8} {'Druglike':<10}")
    print(f"{' -'*50}")

    top_10 = df.nlargest(10, 'QED')
    for i, (_, row) in enumerate(top_10.iterrows()):
        druglike_icon = "✓" if row['Druglike'] else "✗"
        print(f"{i+1:<6} {row['Substitution'][:24]:<25} {row['QED']:<8.3f} "
              f"{row['QED_Improvement']:<+8.3f} {row['MW']:<8.1f} {row['LogP']:<8.2f} "
              f"{druglike_icon:<10}")

    # Worst 10 molecules
    print(f"\n{'='*100}")
    print("WORST 10 MOLECULES BY QED SCORE")
    print(f"{'='*100}")
    print(f"{'Rank':<6} {'Substitution':<25} {'QED':<8} {'ΔQED':<8} {'MW':<8} {'LogP':<8} {'Druglike':<10}")
    print(f"{' -'*50}")

    worst_10 = df.nsmallest(10, 'QED')
    for i, (_, row) in enumerate(worst_10.iterrows()):
        rank = len(df) - i
        druglike_icon = "✓" if row['Druglike'] else "✗"
        print(f"{rank:<6} {row['Substitution'][:24]:<25} {row['QED']:<8.3f} "
              f"{row['QED_Improvement']:<+8.3f} {row['MW']:<8.1f} {row['LogP']:<8.2f} "
              f"{druglike_icon:<10}")

def export_all_results(df, orig_props, filename="all_results_comprehensive.csv"):
    """Export all results to CSV with detailed information"""

    if df.empty:
        print("No data to export")
        return

    # Add classification columns
    df['QED_Class'] = pd.cut(df['QED'],
                             bins=[0, 0.4, 0.5, 0.6, 0.7, 1.0],
                             labels=['Very Poor', 'Poor', 'Moderate', 'Good', 'Excellent'])

    df['Improvement_Class'] = pd.cut(df['QED_Improvement'],
                                     bins=[-np.inf, -0.1, -0.05, 0, 0.05, 0.1, np.inf],
                                     labels=['Major Decline', 'Moderate Decline', 'Minor Decline',
                                             'Minor Improvement', 'Moderate Improvement', 'Major Improvement'])

    # Add property scores
    df['LogP_Score'] = 1 - abs(df['LogP'] - 2) / 5  # Optimal around 2
    df['Size_Score'] = 1 - abs(df['MW'] - 350) / 200  # Optimal around 350
    df['Overall_Score'] = (df['QED'] * 0.5 + df['Druglike'] * 0.2 +
                          df['Synth_Score'] * 0.1 + df['LogP_Score'] * 0.1 +
                          df['Size_Score'] * 0.1)

    # Sort by overall score
    df = df.sort_values('Overall_Score', ascending=False)

    # Export to CSV
    df.to_csv(filename, index=False)
    print(f"\nAll results exported to: {filename}")

    # Also export summary
    summary = df[['SMILES', 'Substitution', 'QED', 'QED_Improvement',
                  'MW', 'LogP', 'Druglike', 'QED_Class', 'Overall_Score']]
    summary.to_csv("summary_results.csv", index=False)
    print(f"Summary exported to: summary_results.csv")

# ---------------
# MAIN EXECUTION
# ---------------
def run_comprehensive_visualization_analysis(original_smiles):
    """Run comprehensive analysis with full visualization"""

    print("="*100)
    print("COMPREHENSIVE FUNCTIONAL GROUP SUBSTITUTION ANALYSIS")
    print("WITH FULL VISUALIZATION OF ALL STRUCTURES")
    print("="*100)

    # Get original properties
    orig_props = compute_all_properties(original_smiles)

    print(f"\nORIGINAL MOLECULE:")
    print(f"SMILES: {original_smiles}")
    print(f"QED: {orig_props['QED']:.3f} | LogP: {orig_props['LogP']:.2f} | MW: {orig_props['MW']:.1f}")
    print(f"HBD: {orig_props['HBD']} | HBA: {orig_props['HBA']} | RotB: {orig_props['RotB']}")
    print(f"Druglike: {orig_props['Druglike']}")

    # Step 1: Generate ALL possible substitutions
    print(f"\n{'='*100}")
    print("STEP 1: GENERATING ALL POSSIBLE SUBSTITUTIONS")
    print(f"{'='*100}")

    all_results = generate_all_possible_substitutions(original_smiles)

    if not all_results:
        print("No valid substitutions generated!")
        return None, orig_props

    # Convert to DataFrame
    df = pd.DataFrame(all_results)

    # Step 2: Display detailed statistics
    print(f"\n{'='*100}")
    print("STEP 2: DETAILED STATISTICAL ANALYSIS")
    print(f"{'='*100}")

    create_detailed_statistics_table(df, orig_props)

    # Step 3: Visualize all structures in grid
    print(f"\n{'='*100}")
    print("STEP 3: VISUALIZING ALL STRUCTURES")
    print(f"{'='*100}")

    visualize_all_structures_grid(df, original_smiles, max_display=36)

    # Step 4: Create comprehensive plots
    print(f"\n{'='*100}")
    print("STEP 4: CREATING COMPREHENSIVE PLOTS")
    print(f"{'='*100}")

    # Plot 1: QED distribution plot
    print("\n1. QED Distribution Analysis:")
    create_qed_distribution_plot(df, orig_props['QED'])

    # Plot 2: Interactive scatter plot
    print("\n2. Interactive Scatter Plot:")
    create_interactive_scatter_plot(df, orig_props)

    # Plot 3: Radar charts for top molecules
    print("\n3. Property Radar Charts (Top Molecules):")
    create_property_radar_charts(df, orig_props, top_n=4)

    # Plot 4: Improvement heatmap
    print("\n4. Substitution Improvement Heatmap:")
    create_improvement_heatmap(df)

    # Step 5: Export all results
    print(f"\n{'='*100}")
    print("STEP 5: EXPORTING ALL RESULTS")
    print(f"{'='*100}")

    export_all_results(df, orig_props)

    # Step 6: Create HTML report
    print(f"\n{'='*100}")
    print("STEP 6: CREATING HTML REPORT")
    print(f"{'='*100}")

    create_html_report(df, orig_props, original_smiles)

    return df, orig_props

def create_html_report(df, orig_props, original_smiles):
    """Create comprehensive HTML report"""

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Comprehensive Molecular Substitution Analysis</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 20px;
                background-color: #f5f5f5;
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
                background-color: white;
                padding: 20px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.1);
            }}
            h1, h2, h3 {{
                color: #2c3e50;
            }}
            .summary-box {{
                background-color: #e8f4f8;
                padding: 15px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .stats-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 15px;
                margin: 20px 0;
            }}
            .stat-card {{
                background-color: #f8f9fa;
                padding: 15px;
                border-radius: 5px;
                text-align: center;
                border-left: 4px solid #3498db;
            }}
            .stat-card.good {{
                border-left-color: #2ecc71;
            }}
            .stat-card.bad {{
                border-left-color: #e74c3c;
            }}
            .stat-card.warning {{
                border-left-color: #f39c12;
            }}
            .stat-value {{
                font-size: 24px;
                font-weight: bold;
                margin: 10px 0;
            }}
            .molecule-table {{
                width: 100%;
                border-collapse: collapse;
                margin: 20px 0;
            }}
            .molecule-table th, .molecule-table td {{
                border: 1px solid #ddd;
                padding: 8px;
                text-align: left;
            }}
            .molecule-table th {{
                background-color: #4CAF50;
                color: white;
            }}
            .molecule-table tr:nth-child(even) {{
                background-color: #f2f2f2;
            }}
            .qed-excellent {{
                background-color: #d4edda !important;
            }}
            .qed-good {{
                background-color: #fff3cd !important;
            }}
            .qed-moderate {{
                background-color: #f8d7da !important;
            }}
            .qed-poor {{
                background-color: #f5c6cb !important;
            }}
            .qed-verypoor {{
                background-color: #721c24 !important;
                color: white;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Comprehensive Molecular Substitution Analysis Report</h1>
            <p>Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

            <div class="summary-box">
                <h2>Original Molecule</h2>
                <p><strong>SMILES:</strong> {original_smiles}</p>
                <p><strong>QED:</strong> {orig_props['QED']:.3f} | <strong>LogP:</strong> {orig_props['LogP']:.2f} |
                   <strong>MW:</strong> {orig_props['MW']:.1f}</p>
                <p><strong>Druglike:</strong> {orig_props['Druglike']} |
                   <strong>Lipinski Violations:</strong> {orig_props['Lipinski_Violations']}</p>
            </div>

            <div class="stats-grid">
                <div class="stat-card good">
                    <h3>Total Molecules</h3>
                    <div class="stat-value">{len(df)}</div>
                    <p>Generated unique structures</p>
                </div>
                <div class="stat-card good">
                    <h3>Best QED</h3>
                    <div class="stat-value">{df['QED'].max():.3f}</div>
                    <p>Improvement: {df['QED'].max() - orig_props['QED']:+.3f}</p>
                </div>
                <div class="stat-card">
                    <h3>Average QED</h3>
                    <div class="stat-value">{df['QED'].mean():.3f}</div>
                    <p>Across all molecules</p>
                </div>
                <div class="stat-card warning">
                    <h3>Druglike %</h3>
                    <div class="stat-value">{len(df[df['Druglike']])/len(df)*100:.1f}%</div>
                    <p>{len(df[df['Druglike']])} of {len(df)} molecules</p>
                </div>
            </div>

            <h2>Top 20 Molecules by QED Score</h2>
            <table class="molecule-table">
                <thead>
                    <tr>
                        <th>Rank</th>
                        <th>Substitution</th>
                        <th>QED</th>
                        <th>ΔQED</th>
                        <th>MW</th>
                        <th>LogP</th>
                        <th>Druglike</th>
                    </tr>
                </thead>
                <tbody>
    """

    # Add top 20 molecules
    top_20 = df.nlargest(20, 'QED')
    for i, (_, row) in enumerate(top_20.iterrows()):
        qed_class = get_qed_class(row['QED'])
        druglike_icon = "✓" if row['Druglike'] else "✗"

        html_content += f"""
                    <tr class="{qed_class}">
                        <td>{i+1}</td>
                        <td>{row['Substitution']}</td>
                        <td>{row['QED']:.3f}</td>
                        <td>{row['QED_Improvement']:+.3f}</td>
                        <td>{row['MW']:.1f}</td>
                        <td>{row['LogP']:.2f}</td>
                        <td>{druglike_icon}</td>
                    </tr>
        """

    html_content += """
                </tbody>
            </table>

            <h2>Key Findings</h2>
            <ul>
                <li><strong>Best substitution:</strong> """

    if not df.empty:
        best = df.loc[df['QED'].idxmax()]
        html_content += f"{best['Substitution']} (QED: {best['QED']:.3f}, Δ{best['QED_Improvement']:+.3f})"

    html_content += """</li>
                <li><strong>Most promising modifications:</strong> Focus on substitutions that maintain or improve QED while keeping molecular weight in the druglike range (200-500 Da)</li>
                <li><strong>Warning:</strong> Some substitutions significantly decrease QED or make molecules non-druglike</li>
            </ul>

            <h2>Recommendations for Further Analysis</h2>
            <ol>
                <li>Validate top candidates with additional ADMET predictions</li>
                <li>Consider synthetic accessibility of proposed modifications</li>
                <li>Test combinations of successful single substitutions</li>
                <li>Perform molecular dynamics simulations for top candidates</li>
            </ol>

            <div class="summary-box">
                <h3>Report Files Generated</h3>
                <ul>
                    <li><strong>all_results_comprehensive.csv</strong> - Complete dataset with all properties</li>
                    <li><strong>summary_results.csv</strong> - Summary of key results</li>
                    <li><strong>visualization_plots.png/jpg</strong> - Various visualization plots</li>
                </ul>
            </div>
        </div>
    </body>
    </html>
    """

    # Save HTML report
    with open("comprehensive_analysis_report.html", "w") as f:
        f.write(html_content)

    print(f"HTML report saved to: comprehensive_analysis_report.html")

    # Display in notebook
    display(HTML(html_content))

def get_qed_class(qed_score):
    """Get CSS class for QED score"""
    if qed_score >= 0.7:
        return "qed-excellent"
    elif qed_score >= 0.6:
        return "qed-good"
    elif qed_score >= 0.5:
        return "qed-moderate"
    elif qed_score >= 0.4:
        return "qed-poor"
    else:
        return "qed-verypoor"

# --------------------------------------
# MINIMAL RL AGENT FOR PATHWAY LEARNING
# --------------------------------------
class PathwayRLAgent:
    """Lightweight RL agent that learns optimal substitution sequences"""

    def __init__(self):
        # Memory for successful pathways
        self.successful_pathways = []
        self.substitution_memory = defaultdict(list)  # (from_smiles, to_smiles) -> success_count
        self.learned_rules = []

    def learn_from_batch(self, df, original_smiles, orig_qed):
        """Learn from a batch of single substitutions"""
        if df.empty:
            return

        # Extract successful substitutions (improved QED)
        successful_subs = df[df['QED_Improvement'] > 0]

        # Learn rules: what substitutions work well from original
        for _, row in successful_subs.iterrows():
            rule = {
                'from_smiles': original_smiles,
                'to_smiles': row['SMILES'],
                'substitution': row['Substitution'],
                'qed_improvement': row['QED_Improvement'],
                'qed_final': row['QED'],
                'properties': {
                    'MW': row['MW'],
                    'LogP': row['LogP'],
                    'Druglike': row['Druglike']
                }
            }
            self.learned_rules.append(rule)

            # Store in memory
            key = (original_smiles, row['SMILES'])
            self.substitution_memory[key].append(row['QED_Improvement'])

        print(f"Learned {len(successful_subs)} successful substitution rules")

    def generate_pathways(self, original_smiles, df, max_steps=3):
        """Generate multi-step pathways using learned rules"""
        if not self.learned_rules:
            print("No rules learned yet")
            return []

        pathways = []

        # Start with original molecule
        start_smiles = original_smiles

        # Find all possible first steps
        first_steps = [r for r in self.learned_rules if r['from_smiles'] == start_smiles]

        for step1 in first_steps[:5]:  # Limit to top 5 first steps
            pathway = [{
                'step': 0,
                'smiles': start_smiles,
                'action': 'Start',
                'qed': compute_all_properties(start_smiles)['QED']
            }, {
                'step': 1,
                'smiles': step1['to_smiles'],
                'action': step1['substitution'],
                'qed': step1['qed_final']
            }]

            # Try to extend pathway
            current_smiles = step1['to_smiles']

            for step_num in range(2, max_steps + 1):
                # Find next possible steps from current molecule
                next_steps = [r for r in self.learned_rules if r['from_smiles'] == current_smiles]

                if not next_steps:
                    break

                # Choose best next step
                next_step = max(next_steps, key=lambda x: x['qed_improvement'])

                pathway.append({
                    'step': step_num,
                    'smiles': next_step['to_smiles'],
                    'action': next_step['substitution'],
                    'qed': next_step['qed_final']
                })

                current_smiles = next_step['to_smiles']

            pathways.append({
                'pathway': pathway,
                'total_improvement': pathway[-1]['qed'] - pathway[0]['qed'],
                'steps': len(pathway) - 1
            })

        # Sort by total improvement
        pathways.sort(key=lambda x: x['total_improvement'], reverse=True)

        # Keep only unique pathways (different final molecules)
        unique_pathways = []
        seen_final_smiles = set()

        for pathway in pathways:
            final_smiles = pathway['pathway'][-1]['smiles']
            if final_smiles not in seen_final_smiles:
                seen_final_smiles.add(final_smiles)
                unique_pathways.append(pathway)

        self.successful_pathways = unique_pathways[:5]  # Keep top 5
        return self.successful_pathways

    def display_learned_pathways(self):
        """Display the learned pathways"""
        if not self.successful_pathways:
            print("No pathways learned yet")
            return

        print(f"\n{'='*100}")
        print(f"LEARNED OPTIMAL PATHWAYS ({len(self.successful_pathways)} unique)")
        print(f"{'='*100}")

        for i, pathway_info in enumerate(self.successful_pathways):
            pathway = pathway_info['pathway']
            print(f"\nPathway #{i+1}:")
            print(f"Total QED Improvement: {pathway_info['total_improvement']:.3f}")
            print(f"Steps: {pathway_info['steps']}")
            print("-" * 80)

            for step in pathway:
                print(f"Step {step['step']}: {step['action']}")
                print(f"  QED: {step['qed']:.3f}")
                if step['step'] > 0:
                    prev_qed = pathway[step['step']-1]['qed']
                    print(f"  ΔQED: {step['qed'] - prev_qed:+.3f}")
                print()

            # Visualize the pathway molecules
            self.visualize_pathway(pathway)

    def visualize_pathway(self, pathway):
        """Visualize a single pathway"""
        mols = []
        legends = []

        for step in pathway:
            mol = Chem.MolFromSmiles(step['smiles'])
            if mol:
                mols.append(mol)

                # Create legend
                legend = f"Step {step['step']}"
                if step['action'] != 'Start':
                    legend += f": {step['action']}"
                legend += f"\nQED: {step['qed']:.3f}"

                if step['step'] > 0:
                    prev_qed = pathway[step['step']-1]['qed']
                    legend += f" (Δ{step['qed'] - prev_qed:+.3f})"

                legends.append(legend)

        if mols:
            # Display molecules in a row
            img = Draw.MolsToGridImage(
                mols,
                molsPerRow=len(mols),
                subImgSize=(300, 300),
                legends=legends,
                returnPNG=False
            )
            display(img)

    def save_learned_rules(self, filename="learned_pathways.json"):
        """Save learned rules to file"""
        data = {
            'successful_pathways': self.successful_pathways,
            'learned_rules': self.learned_rules,
            'timestamp': datetime.now().isoformat()
        }

        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)

        print(f"Learned pathways saved to {filename}")

    def load_learned_rules(self, filename="learned_pathways.json"):
        """Load learned rules from file"""
        try:
            with open(filename, 'r') as f:
                data = json.load(f)

            self.successful_pathways = data.get('successful_pathways', [])
            self.learned_rules = data.get('learned_rules', [])

            # Rebuild substitution memory
            self.substitution_memory.clear()
            for rule in self.learned_rules:
                key = (rule['from_smiles'], rule['to_smiles'])
                if key not in self.substitution_memory:
                    self.substitution_memory[key] = []
                self.substitution_memory[key].append(rule['qed_improvement'])

            print(f"Loaded {len(self.learned_rules)} learned rules")
            return True
        except Exception as e:
            print(f"Error loading rules: {e}")
            return False


# ------------------------------------
# INTEGRATE RL WITH EXISTING ANALYSIS
# ------------------------------------
def run_analysis_with_rl(original_smiles):
    """Run your existing analysis with RL pathway learning"""

    print("\n" + "="*100)
    print("RUNNING ANALYSIS WITH RL PATHWAY LEARNING")
    print("="*100)

    # Run your existing comprehensive analysis
    results_df, original_props = run_comprehensive_visualization_analysis(original_smiles)

    if results_df is None or results_df.empty:
        print("No results generated")
        return None, None, None

    print(f"\n{'='*100}")
    print("LEARNING OPTIMAL PATHWAYS FROM RESULTS")
    print(f"{'='*100}")

    # Initialize RL agent
    rl_agent = PathwayRLAgent()

    # Learn from the batch results
    rl_agent.learn_from_batch(results_df, original_smiles, original_props['QED'])

    # Generate pathways
    pathways = rl_agent.generate_pathways(original_smiles, results_df, max_steps=3)

    # Display learned pathways
    if pathways:
        rl_agent.display_learned_pathways()

        # Compare with single-step results
        best_single = results_df.loc[results_df['QED'].idxmax()]
        best_pathway = pathways[0]  # Best multi-step pathway

        print(f"\n{'='*100}")
        print("COMPARISON: SINGLE vs MULTI-STEP OPTIMIZATION")
        print(f"{'='*100}")

        print(f"\nBest Single Substitution:")
        print(f"  Substitution: {best_single['Substitution']}")
        print(f"  QED: {best_single['QED']:.3f}")
        print(f"  Improvement: {best_single['QED_Improvement']:+.3f}")

        print(f"\nBest Multi-Step Pathway ({best_pathway['steps']} steps):")
        print(f"  Final QED: {best_pathway['pathway'][-1]['qed']:.3f}")
        print(f"  Total Improvement: {best_pathway['total_improvement']:+.3f}")

        # Check if pathway is better
        if best_pathway['total_improvement'] > best_single['QED_Improvement']:
            print(f"\n✓ PATHWAY IS BETTER by {best_pathway['total_improvement'] - best_single['QED_Improvement']:+.3f} QED")
            print("  Multi-step optimization found superior result!")
        else:
            print(f"\n✓ Single substitution gives immediate best result")
            print("  Pathway learning still valuable for exploring alternatives")

        # Save learned pathways
        rl_agent.save_learned_rules()
    else:
        print("No pathways could be generated")

    return results_df, original_props, rl_agent


# ------------------------
# ENHANCED MAIN EXECUTION
# ------------------------
if __name__ == "__main__":
    print("\n" + "="*100)
    print("ENHANCED MOLECULAR OPTIMIZATION WITH PATHWAY LEARNING")
    print("="*100)

    # Run enhanced analysis with RL
    results, props, agent = run_analysis_with_rl(true_smiles)

    if agent and agent.successful_pathways:
        print(f"\n{'='*100}")
        print("ANALYSIS COMPLETE WITH PATHWAY LEARNING!")
        print(f"{'='*100}")

        print(f"\nKey achievements:")
        print(f"1. Generated {len(results)} single substitutions")
        print(f"2. Learned {len(agent.learned_rules)} successful substitution rules")
        print(f"3. Discovered {len(agent.successful_pathways)} optimal pathways")
        print(f"4. Best pathway improvement: {agent.successful_pathways[0]['total_improvement']:.3f}")

        print(f"\nGenerated files:")
        print(f"1. all_results_comprehensive.csv - All single substitutions")
        print(f"2. learned_pathways.json - Learned multi-step pathways")
        print(f"3. comprehensive_analysis_report.html - Full analysis report")
    else:
        print("\nAnalysis completed without pathway learning")

    print("\n" + "="*100)

In [ ]:

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# PHASE 4.3 — FragmentBERT with RL Optimization
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

# --------
# Imports
# --------
!pip install stable_baselines3
import re
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict, defaultdict
from typing import List, Tuple, Dict, Any
import gymnasium as gym
from gymnasium import spaces
from transformers import pipeline, AutoTokenizer, AutoModelForMaskedLM
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, AllChem
from rdkit.Chem.QED import qed
from rdkit.Chem.Scaffolds import MurckoScaffold
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Display for visualization
try:
    from IPython.display import display
except:
    display = print

# ----------------------------
# Paths - USING TRAINED MODEL
# ----------------------------
MODEL_PATH = "/content/FragmentBERT/checkpoints/final_model"
TOKENIZER_PATH = "/content/FragmentBERT"  # Tokenizer directory

print(f"Model path: {MODEL_PATH}")
print(f"Tokenizer path: {TOKENIZER_PATH}")

# Check if files exist
if os.path.exists(MODEL_PATH):
    print(f"✓ Model directory exists")
    model_files = os.listdir(MODEL_PATH)
    print(f"  Files found: {model_files}")
else:
    print(f"✗ Model directory not found: {MODEL_PATH}")

# ----------------------------------
# SIMPLE TOKENIZER FOR MOSES SMILES
# ----------------------------------

class SimpleSMILESTokenizer:
    """Simple tokenizer that matches Moses encoding"""

    def __init__(self):
        self.mask_token = "[MASK]"
        self.cls_token = "[CLS]"
        self.sep_token = "[SEP]"
        self.pad_token = "[PAD]"
        self.unk_token = "[UNK]"

        # Moses SMILES regex (same as in your Phase 1)
        self.regex = re.compile(
            r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\||\(|\)|\.|=|#|-|\+|\\\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"
        )

        print("✓ SimpleSMILESTokenizer initialized")

    def tokenize(self, text):
        """Tokenize SMILES string"""
        if not text or not isinstance(text, str):
            return []

        tokens = self.regex.findall(text)
        # Clean empty tokens
        return [t for t in tokens if t.strip()]

    def encode(self, text):
        """Encode text to token IDs (simple hash-based)"""
        tokens = self.tokenize(text)
        return [hash(t) % 10000 for t in tokens]

    def decode(self, token_ids, skip_special_tokens=True):
        """Decode token IDs back to string"""
        # For simplicity, just return placeholder
        return "[DECODED_SMILES]"

# -----------------------------
# MODEL LOADER - FIXED VERSION
# -----------------------------

def load_fragmentbert_model():
    """Load FragmentBERT model with proper error handling"""

    print("\n" + "="*60)
    print("LOADING FRAGMENTBERT MODEL")
    print("="*60)

    try:
        # Check if model exists
        if not os.path.exists(MODEL_PATH):
            print(f"✗ Model path not found: {MODEL_PATH}")
            print("Trying public model...")
            return load_public_model()

        # Check for required files
        required_files = ['pytorch_model.bin', 'config.json', 'tokenizer.json']
        existing_files = os.listdir(MODEL_PATH) if os.path.exists(MODEL_PATH) else []

        print(f"Checking model files in {MODEL_PATH}:")
        for file in required_files:
            if file in existing_files:
                print(f"  ✓ {file}")
            else:
                print(f"  ✗ {file} (missing)")

        # Try to load the model
        print("\nLoading model...")
        model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)

        # Try to load tokenizer
        tokenizer = None
        tokenizer_paths = [
            MODEL_PATH,  # Check in model directory
            TOKENIZER_PATH,  # Check in tokenizer directory
        ]

        for path in tokenizer_paths:
            if os.path.exists(os.path.join(path, "tokenizer.json")):
                try:
                    tokenizer = AutoTokenizer.from_pretrained(path)
                    print(f"✓ Tokenizer loaded from {path}")
                    break
                except Exception as e:
                    print(f"  Could not load tokenizer from {path}: {e}")

        if tokenizer is None:
            print(" Could not load HuggingFace tokenizer, using simple tokenizer")
            tokenizer = SimpleSMILESTokenizer()
            # Create adapter for pipeline
            class TokenizerAdapter:
                def __init__(self, simple_tokenizer):
                    self.simple_tokenizer = simple_tokenizer
                    self.mask_token = simple_tokenizer.mask_token

                def __call__(self, text, **kwargs):
                    return self.simple_tokenizer.encode(text)

            tokenizer = TokenizerAdapter(tokenizer)

        # Create pipeline
        fill_mask = pipeline(
            "fill-mask",
            model=model,
            tokenizer=tokenizer,
            device=0 if torch.cuda.is_available() else -1
        )

        print("\n✓ FragmentBERT loaded successfully!")

        # Test the model
        print("\nTesting model with simple example...")
        try:
            test_input = "C [MASK] O"
            results = fill_mask(test_input, top_k=3)
            print(f"Input: {test_input}")
            for i, result in enumerate(results):
                print(f"  {i+1}. {result['sequence']} (score: {result['score']:.3f})")
        except Exception as e:
            print(f"  Test error (but continuing): {e}")

        return fill_mask

    except Exception as e:
        print(f"✗ Error loading model: {e}")
        print("\nFalling back to public model...")
        return load_public_model()

def load_public_model():
    """Load public FragmentBERT model as fallback"""
    try:
        print("Loading public FragmentBERT model...")
        fill_mask = pipeline(
            "fill-mask",
            model="gokceuludogan/FragmentBERT",
            tokenizer="gokceuludogan/FragmentBERT"
        )
        print("✓ Public FragmentBERT loaded")
        return fill_mask
    except Exception as e:
        print(f"✗ Error loading public model: {e}")
        print("Creating mock pipeline for testing...")
        return create_mock_pipeline()

def create_mock_pipeline():
    """Create mock pipeline for testing"""
    print("Creating mock pipeline...")

    class MockPipeline:
        def __init__(self):
            self.mask_token = "[MASK]"
            print("✓ Mock pipeline created")

        def __call__(self, text, top_k=5):
            print(f"  Mock generation: {text[:50]}...")

            # Simple mock responses
            responses = []
            tokens = text.split()

            for i in range(top_k):
                # Replace [MASK] with random tokens
                filled = []
                for token in tokens:
                    if token == "[MASK]":
                        filled.append(random.choice(["C", "N", "O", "Cl", "(", ")", "=", "#"]))
                    else:
                        filled.append(token)

                sequence = " ".join(filled)
                responses.append({
                    "sequence": sequence,
                    "score": 0.9 - (i * 0.1),
                    "token_str": filled[-1] if filled else ""
                })

            return responses

    return MockPipeline()

# ----------------------------
# SMILES PROCESSING FUNCTIONS
# ----------------------------

def moses_encode(smiles: str):
    """Tokenize SMILES string with Moses regex"""
    if not smiles or not isinstance(smiles, str):
        return None

    # Moses SMILES regex (same as your Phase 1)
    SMILES_REGEX = re.compile(
        r"\[[^\]]+\]|Br|Cl|Si|Na|Li|Mg|Ca|"
        r"[A-Z][a-z]?|\d+|=|#|-|\(|\)|\.|@|\+|\\|/"
    )

    smiles = smiles.strip()
    if len(smiles) < 2:
        return None

    tokens = SMILES_REGEX.findall(smiles)
    if len(tokens) < 2:
        return None

    return " ".join(tokens)

def moses_decode(tokenized: str):
    """Convert tokenized SMILES back to string"""
    return tokenized.replace(" ", "")

def validate_and_fix_smiles(smiles: str):
    """Validate and fix invalid SMILES"""
    if not smiles or not isinstance(smiles, str):
        return None

    # Try direct parsing
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        try:
            canon = Chem.MolToSmiles(mol, canonical=True)
            if Chem.MolFromSmiles(canon):
                return canon
        except:
            pass

    # Try to fix common issues
    fixed = smiles

    # Fix bracket atoms
    fixes = [
        ('[C]', 'C'), ('[c]', 'c'), ('[N]', 'N'), ('[n]', 'n'),
        ('[O]', 'O'), ('[o]', 'o'), ('[S]', 'S'), ('[s]', 's'),
        ('[P]', 'P'), ('[p]', 'p'), ('[F]', 'F'), ('[Cl]', 'Cl'),
        ('[Br]', 'Br'), ('[I]', 'I'), ('[H]', 'H')
    ]

    for old, new in fixes:
        fixed = fixed.replace(old, new)

    # Fix duplicate parentheses
    fixed = re.sub(r'\(\(', '(', fixed)
    fixed = re.sub(r'\)\)', ')', fixed)

    # Try parsing fixed version
    mol = Chem.MolFromSmiles(fixed)
    if mol:
        try:
            canon = Chem.MolToSmiles(mol, canonical=True)
            if Chem.MolFromSmiles(canon):
                return canon
        except:
            pass

    return None

def compute_druglikeness_score(smiles: str) -> Dict[str, Any]:
    """Compute drug-likeness metrics"""
    valid_smiles = validate_and_fix_smiles(smiles)
    if not valid_smiles:
        return None

    mol = Chem.MolFromSmiles(valid_smiles)
    if mol is None:
        return None

    try:
        mw = Descriptors.MolWt(mol)
        logp = Crippen.MolLogP(mol)
        hbd = Lipinski.NumHDonors(mol)
        hba = Lipinski.NumHAcceptors(mol)
        rotb = Lipinski.NumRotatableBonds(mol)
        tpsa = Descriptors.TPSA(mol)
        qed_val = qed(mol)
        ring_count = Descriptors.RingCount(mol)

        # Lipinski violations
        lipinski_violations = 0
        if mw > 500: lipinski_violations += 1
        if logp > 5: lipinski_violations += 1
        if hbd > 5: lipinski_violations += 1
        if hba > 10: lipinski_violations += 1

        # Drug score (simple version)
        lipinski_score = 1.0 - (lipinski_violations * 0.25)

        # Synthetic accessibility (simple estimate)
        sa_score = 0.0
        num_atoms = mol.GetNumAtoms()
        if num_atoms > 50: sa_score += 0.3
        elif num_atoms > 30: sa_score += 0.1

        rings = mol.GetRingInfo().NumRings()
        if rings > 5: sa_score += 0.2
        elif rings > 3: sa_score += 0.1

        sa_score = min(sa_score, 1.0)

        # Combined drug score
        drug_score = (
            0.4 * lipinski_score +
            0.4 * qed_val +
            0.2 * (1.0 - sa_score)
        )

        return {
            "SMILES": valid_smiles,
            "MW": mw,
            "LogP": logp,
            "HBD": hbd,
            "HBA": hba,
            "RotB": rotb,
            "TPSA": tpsa,
            "QED": qed_val,
            "RingCount": ring_count,
            "DrugScore": drug_score,
            "LipinskiViolations": lipinski_violations,
            "IsDruglike": lipinski_violations <= 1 and qed_val >= 0.4
        }
    except Exception as e:
        print(f"Error computing drug properties: {e}")
        return None

# --------------------------
# SIMPLIFIED RL ENVIRONMENT
# --------------------------

class SimpleRLEnv(gym.Env):
    """Simplified RL environment for molecule optimization"""

    def __init__(self, base_smiles, fill_mask_pipeline, max_steps=3):
        super().__init__()

        print(f"\nInitializing RL Environment for: {base_smiles[:50]}...")

        # Validate SMILES
        self.base_smiles = validate_and_fix_smiles(base_smiles)
        if not self.base_smiles:
            raise ValueError(f"Invalid base SMILES: {base_smiles}")

        print(f"✓ Base SMILES: {self.base_smiles}")

        self.fill_mask = fill_mask_pipeline
        self.max_steps = max_steps

        # Tokenize using Moses encoding
        encoded = moses_encode(self.base_smiles)
        if not encoded:
            raise ValueError(f"Could not tokenize SMILES: {self.base_smiles}")

        self.tokens = encoded.split()
        self.num_tokens = len(self.tokens)

        print(f"✓ Tokenized into {self.num_tokens} tokens")
        print(f"  Tokens: {self.tokens[:10]}...")

        # Action space: which token to mask
        self.action_space = spaces.Discrete(self.num_tokens)

        # Observation space: simple features
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(self.num_tokens + 5,),
            dtype=np.float32
        )

        # Get initial score
        self.initial_props = compute_druglikeness_score(self.base_smiles)
        self.best_score = self.initial_props['DrugScore'] if self.initial_props else 0.0
        self.best_smiles = self.base_smiles

        print(f"✓ Initial drug score: {self.best_score:.3f}")

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.current_smiles = self.base_smiles
        self.current_tokens = self.tokens.copy()
        self.mask_state = np.zeros(self.num_tokens, dtype=np.float32)

        print(f"  Environment reset")
        return self._get_observation(), {}

    def _get_observation(self):
        """Get observation vector"""
        # Token mask state
        token_features = list(self.mask_state)

        # Molecular properties
        props = compute_druglikeness_score(self.current_smiles)
        mol_features = [0.0] * 5
        if props:
            mol_features = [
                props['MW'] / 1000.0,
                (props['LogP'] + 5) / 10.0,
                props['DrugScore'],
                props['QED'],
                float(props['LipinskiViolations']) / 4.0
            ]

        return np.array(token_features + mol_features, dtype=np.float32)

    def step(self, action):
        self.current_step += 1

        print(f"\n  Step {self.current_step}: Action = {action}")

        # Validate action
        if action < 0 or action >= self.num_tokens:
            return self._get_observation(), -1.0, False, False, {'error': 'Invalid action'}

        # Check if already masked
        if self.mask_state[action] == 1.0:
            return self._get_observation(), -0.5, False, False, {'warning': 'Already masked'}

        # Apply mask
        self.mask_state[action] = 1.0
        print(f"    Masking token {action}: '{self.current_tokens[action]}'")

        # Create masked input
        masked_tokens = []
        for i, token in enumerate(self.current_tokens):
            if self.mask_state[i] == 1.0:
                masked_tokens.append("[MASK]")
            else:
                masked_tokens.append(token)

        masked_input = " ".join(masked_tokens)
        print(f"    Masked input: {masked_input[:60]}...")

        # Generate new molecule
        new_smiles = None
        try:
            outputs = self.fill_mask(masked_input, top_k=3)

            for output in outputs:
                generated = output['sequence']
                # Remove spaces for SMILES
                generated = generated.replace(' ', '')
                print(f"    Generated: {generated[:50]}... (score: {output['score']:.3f})")

                # Validate
                valid = validate_and_fix_smiles(generated)
                if valid:
                    new_smiles = valid
                    print(f"    Validated: {new_smiles[:50]}...")
                    break

        except Exception as e:
            print(f"    Generation error: {e}")
            # Fallback: simple mutation
            new_tokens = self.current_tokens.copy()
            if action < len(new_tokens):
                new_tokens[action] = random.choice(['C', 'N', 'O', 'Cl', 'F'])
                new_smiles = ''.join(new_tokens)
                print(f"    Fallback: {new_smiles[:50]}...")

        if new_smiles and new_smiles != self.current_smiles:
            # Calculate reward
            reward, info = self._calculate_reward(new_smiles)

            # Update state
            self.current_smiles = new_smiles

            # Re-tokenize
            encoded = moses_encode(new_smiles)
            if encoded:
                self.current_tokens = encoded.split()
                self.num_tokens = len(self.current_tokens)
                self.mask_state = np.zeros(self.num_tokens, dtype=np.float32)

            terminated = self.current_step >= self.max_steps

            print(f"    Reward: {reward:.3f}, New score: {info.get('score', 0):.3f}")

            return self._get_observation(), reward, terminated, False, info
        else:
            print("    Generation failed")
            reward = -1.0
            terminated = self.current_step >= self.max_steps
            return self._get_observation(), reward, terminated, False, {'error': 'Generation failed'}

    def _calculate_reward(self, new_smiles):
        """Calculate reward for new molecule"""
        props = compute_druglikeness_score(new_smiles)

        if not props:
            return -0.5, {'error': 'Invalid molecule'}

        current_score = props['DrugScore']
        improvement = current_score - self.best_score

        # Base reward
        reward = current_score

        # Bonus for improvement
        if improvement > 0:
            reward += improvement * 3.0
            self.best_score = current_score
            self.best_smiles = new_smiles

        # Penalty for violations
        if props['LipinskiViolations'] > 1:
            reward -= 0.3

        info = {
            'drug_props': props,
            'score': current_score,
            'improvement': improvement,
            'smiles': new_smiles
        }

        return reward, info

    def render(self):
        print(f"\n--- RL Environment ---")
        print(f"Step: {self.current_step}/{self.max_steps}")
        print(f"Current: {self.current_smiles[:60]}...")
        print(f"Best score: {self.best_score:.3f}")

# -----------------------
# SIMPLE RL OPTIMIZATION
# -----------------------

def simple_rl_optimization(base_smiles, fill_mask_pipeline, num_episodes=3, max_steps=2):
    """Simple RL optimization that works reliably"""

    print(f"\n{'='*60}")
    print(f"RL OPTIMIZATION")
    print(f"Target: {base_smiles[:50]}...")
    print(f"{'='*60}")

    # Create environment
    env = SimpleRLEnv(base_smiles, fill_mask_pipeline, max_steps=max_steps)

    # Simple RL agent
    print("\nSetting up RL agent...")
    model = PPO(
        "MlpPolicy",
        DummyVecEnv([lambda: env]),
        learning_rate=1e-3,
        n_steps=64,
        batch_size=32,
        n_epochs=3,
        gamma=0.9,
        verbose=0
    )

    # Quick training
    try:
        print("Training RL agent...")
        model.learn(total_timesteps=100)
        print("✓ RL training completed")
    except Exception as e:
        print(f"⚠ RL training error: {e}")
        print("  Continuing with exploration...")

    # Run episodes
    best_smiles = base_smiles
    best_score = 0.0

    initial_props = compute_druglikeness_score(base_smiles)
    if initial_props:
        best_score = initial_props['DrugScore']

    print(f"\nRunning {num_episodes} optimization episodes...")

    for episode in range(num_episodes):
        obs, _ = env.reset()
        episode_best_score = best_score
        episode_best_smiles = best_smiles

        for step in range(max_steps):
            try:
                # Mix model predictions with random exploration
                if random.random() > 0.3:  # 70% use model
                    action, _ = model.predict(obs, deterministic=False)
                else:  # 30% random
                    action = [random.randint(0, env.num_tokens - 1)]

                obs, reward, terminated, truncated, info = env.step(action[0])

                # Track best
                if 'score' in info and info['score'] > episode_best_score:
                    episode_best_score = info['score']
                    episode_best_smiles = info.get('smiles', episode_best_smiles)

                if terminated or truncated:
                    break

            except Exception as e:
                print(f"  Episode {episode+1}, Step {step+1}: Error - {e}")
                break

        # Update global best
        if episode_best_score > best_score:
            best_score = episode_best_score
            best_smiles = episode_best_smiles

        print(f"  Episode {episode+1}: Best Score = {episode_best_score:.3f}")

    print(f"\n OPTIMIZATION COMPLETE")
    print(f"{'='*60}")
    print(f"Original: {base_smiles[:50]}...")
    print(f"Original Score: {initial_props['DrugScore'] if initial_props else 'N/A':.3f}")
    print(f"\nOptimized: {best_smiles[:50]}...")

    best_props = compute_druglikeness_score(best_smiles)
    if best_props:
        print(f"Optimized Score: {best_props['DrugScore']:.3f}")
        if initial_props:
            improvement = best_props['DrugScore'] - initial_props['DrugScore']
            print(f"Improvement: {improvement:+.3f}")

    return best_smiles, best_props['DrugScore'] if best_props else 0.0

# -------------------------------
# FRAGMENT OPTIMIZATION PIPELINE
# -------------------------------

def enhanced_fragment_optimization(
    masked_smiles,
    use_rl=True,
    rl_episodes=2,
    top_k=8,
    output_csv="results.csv",
    title="Fragment Optimization"
):
    """Complete optimization pipeline"""

    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")

    # Load model
    fill_mask = load_fragmentbert_model()

    # Process masked SMILES
    print(f"\n[1] Processing masked SMILES: {masked_smiles}")

    # Replace X with [MASK]
    bert_input = masked_smiles.replace('X', '[MASK]')

    # Step 1: FragmentBERT generation
    print(f"\n[2] FragmentBERT generation (top_k={top_k})...")

    fragmentbert_results = []

    try:
        outputs = fill_mask(bert_input, top_k=top_k)
        print(f"  Generated {len(outputs)} candidates")

        for i, output in enumerate(outputs):
            generated = output['sequence']
            # Remove spaces for SMILES
            generated = generated.replace(' ', '')

            print(f"  Candidate {i+1}: {generated[:50]}... (score: {output['score']:.3f})")

            valid = validate_and_fix_smiles(generated)
            if valid:
                props = compute_druglikeness_score(valid)
                if props and props['IsDruglike']:
                    fragmentbert_results.append({
                        **props,
                        'Source': 'FragmentBERT',
                        'GenerationScore': output['score']
                    })
                    print(f"    ✓ Drug-like molecule")

        print(f"  ✓ Found {len(fragmentbert_results)} drug-like molecules")

    except Exception as e:
        print(f"  ✗ Generation error: {e}")

    # Step 2: RL optimization
    rl_results = []
    if use_rl and masked_smiles.count('X') <= 2:
        print(f"\n[3] RL optimization ({rl_episodes} episodes)...")

        # Create base for RL
        base_for_rl = masked_smiles.replace('X', 'C')
        valid_base = validate_and_fix_smiles(base_for_rl)

        if valid_base:
            try:
                rl_smiles, rl_score = simple_rl_optimization(
                    valid_base,
                    fill_mask,
                    num_episodes=rl_episodes,
                    max_steps=2
                )

                rl_props = compute_druglikeness_score(rl_smiles)
                if rl_props and rl_props['IsDruglike']:
                    rl_results.append({
                        **rl_props,
                        'Source': 'RL',
                        'GenerationScore': 1.0
                    })
                    print(f"  ✓ RL generated optimized molecule")
            except Exception as e:
                print(f"  ✗ RL optimization failed: {e}")

    # Combine results
    all_results = fragmentbert_results + rl_results

    if not all_results:
        print("\n✗ No drug-like molecules found")
        return pd.DataFrame()

    # Create DataFrame
    df = pd.DataFrame(all_results)
    df = df.sort_values('DrugScore', ascending=False).reset_index(drop=True)

    # Save results
    df.to_csv(output_csv, index=False)
    print(f"\n[4] Saved {len(df)} molecules to {output_csv}")

    # Display results
    print(f"\n TOP RESULTS:")
    display_cols = ['SMILES', 'DrugScore', 'MW', 'LogP', 'QED', 'Source']
    print(df[display_cols].head(5).to_string(index=False))

    # Try visualization
    try:
        top_mols = []
        legends = []

        for i, row in df.head(min(6, len(df))).iterrows():
            mol = Chem.MolFromSmiles(row['SMILES'])
            if mol:
                top_mols.append(mol)
                legends.append(f"{i+1}: {row['Source']} {row['DrugScore']:.2f}")

        if top_mols:
            img = Draw.MolsToGridImage(
                top_mols,
                molsPerRow=min(3, len(top_mols)),
                legends=legends,
                subImgSize=(250, 150)
            )
            display(img)
    except Exception as e:
        print(f"  Visualization skipped: {e}")

    return df

# ---------------
# MAIN EXECUTION
# ---------------

if __name__ == "__main__":
    print("\n" + "="*60)
    print("FRAGMENT OPTIMIZATION WITH RL")
    print("="*60)
    print(f"Using model from: {MODEL_PATH}")

    # Test molecule
    true_smiles = "CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl"

    # Original properties
    orig_props = compute_druglikeness_score(true_smiles)
    if orig_props:
        print(f"\n🧪 TARGET MOLECULE:")
        print(f"  SMILES: {true_smiles[:60]}...")
        print(f"  Drug Score: {orig_props['DrugScore']:.3f}")
        print(f"  MW: {orig_props['MW']:.1f}, LogP: {orig_props['LogP']:.2f}")

    # Optimization strategies
    strategies = [
        {
            'name': 'Alkoxy Optimization',
            'masked': 'XOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl',
            'use_rl': True
        },
        {
            'name': 'Linker Modification',
            'masked': 'CCOXC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)Cl',
            'use_rl': True
        },
        {
            'name': 'Halogen Replacement',
            'masked': 'CCOC1=CC=C(C=C1)CC2=C(C=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)X',
            'use_rl': True
        }
    ]

    # Run optimizations
    all_results = []

    for strategy in strategies:
        print(f"\n{'='*60}")
        print(f"STRATEGY: {strategy['name']}")
        print(f"{'='*60}")

        df = enhanced_fragment_optimization(
            masked_smiles=strategy['masked'],
            use_rl=strategy['use_rl'],
            rl_episodes=2,
            top_k=6,
            output_csv=f"results_{strategy['name'].lower().replace(' ', '_')}.csv",
            title=strategy['name']
        )

        if not df.empty:
            best = df.iloc[0]
            all_results.append({
                'Strategy': strategy['name'],
                'SMILES': best['SMILES'][:40] + '...',
                'DrugScore': f"{best['DrugScore']:.3f}",
                'Improvement': f"{best['DrugScore'] - orig_props['DrugScore']:.3f}" if orig_props else 'N/A',
                'Method': best['Source'],
                'MW': f"{best['MW']:.1f}",
                'LogP': f"{best['LogP']:.2f}"
            })

    # Final summary
    if all_results:
        print(f"\n{'='*60}")
        print("FINAL SUMMARY")
        print(f"{'='*60}")

        summary_df = pd.DataFrame(all_results)
        print(summary_df.to_string(index=False))

        # Find best
        try:
            best_overall = max(all_results, key=lambda x: float(x['DrugScore']))
            print(f"\n BEST STRATEGY: {best_overall['Strategy']}")
            print(f"   Score: {best_overall['DrugScore']}")
            print(f"   Improvement: {best_overall['Improvement']}")
        except:
            print("\n Could not determine best strategy")

    print(f"\n{'='*60}")
    print("OPTIMIZATION COMPLETE")
    print(f"{'='*60}")